In [ ]:
#################################################################
### Compare metrics across all strategies for the given model ###
#################################################################

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple, Literal

# Allow importing eval.metrics from the scripts/ directory
sys.path.insert(0, str(Path.cwd().parent / "scripts"))
# Available for instance-level analysis:
from eval.metrics import read_traj_metrics

In [ ]:
from analysis.dashboards import (
    build_single_report,
    build_comparison_report,
)
from analysis.data_loading import (
    load_metrics,
    load_instance_observations,
    load_instance_observations_by_id,
)
from analysis.stat_significance import (
    _STAT_AGENT_FIELDS,
    get_stat_significance_overall_df,
    get_stat_significance_per_instance_df,
    get_per_instance_pass_rate_df,
)
from analysis.cross_eval import build_cross_eval_comparison_df
from analysis.comparison_nway import (
    STRATEGY_STYLES,
    plot_pass_rate_nway,
    plot_pooled_metrics_nway,
    build_resolution_table,
    build_compact_stat_sig_df,
    display_compact_stat_sig,
)

In [ ]:
### MODELS ###
GPT_5_4 = "GPT-5.4"
SONNET_4_6 = "Claude Sonnet 4.6"

In [ ]:
##########################################################
### SELECT THE MODEL WHICH TO BUILD THE COMPARISON FOR ###
##########################################################

DESIRED_MODEL = GPT_5_4 # SONNET_4_6

In [ ]:
@dataclass
class Strategy:
    name: str
    display_name: str
    filepath: Path
    model: Literal["GPT-5.4", "Claude Sonnet 4.6"]


def result(relative_filepath: str) -> Path:
    results_filepath_prefix = "/Users/vartiukhov/dev/studies/hse/thesis/thesis-metamorphic-eval/artifacts/results/eval"
    return Path(results_filepath_prefix) / relative_filepath

In [ ]:
# GPT-5.4
strategies_gpt: List[Strategy] = [
    Strategy(
        name="s0-original",
        display_name="s0-original",
        filepath=result("s0-original/java_20_gpt5.4_cost_3.0_runs_5"),
        model=GPT_5_4,
    ),
    Strategy(
        name="s1-renaming",
        display_name="s1-renaming",
        # swap to java_20_s1_2nd_gpt5.4_cost_3.0_runs_5 or java_20_s1_3rd_gpt5.4_cost_3.0_runs_5 to compare variants
        filepath=result("s1-renaming/java_20_s1_gpt5.4_cost_3.0_runs_5"),
        model=GPT_5_4,
    ),
    Strategy(
        name="s2-structural",
        display_name="s2-structural",
        filepath=result("s2-structural/java_20_s2_gpt5.4_cost_3.0_runs_5"),
        model=GPT_5_4,
    ),
    Strategy(
        name="s3-problem-statement",
        display_name="s3-problem-statement",
        filepath=result("s3-problem-statement/java_20_s3_gpt5.4_cost_3.0_runs_5"),
        model=GPT_5_4,
    ),
    Strategy(
        name="s4-combined",
        display_name="s4-combined",
        filepath=result("s4-combined/java_20_s4_gpt5.4_cost_3.0_runs_5"),
        model=GPT_5_4,
    ),
]

# Claude Sonnet 4.6
strategies_sonnet: List[Strategy] = [
    Strategy(
        name="s0-original",
        display_name="s0-original",
        filepath=result("s0-original/java_20_sonnet4.6_cost_3.0_runs_5"),
        model=SONNET_4_6,
    ),
    Strategy(
        name="s1-renaming",
        display_name="s1-renaming",
        filepath=result("s1-renaming/java_20_s1_sonnet4.6_cost_3.0_runs_5"),
        model=SONNET_4_6,
    ),
    Strategy(
        name="s2-structural",
        display_name="s2-structural",
        filepath=result("s2-structural/java_20_s2_sonnet4.6_cost_3.0_runs_5"),
        model=SONNET_4_6,
    ),
    Strategy(
        name="s3-problem-statement",
        display_name="s3-problem-statement",
        filepath=result("s3-problem-statement/java_20_s3_sonnet4.6_cost_3.0_runs_5"),
        model=SONNET_4_6,
    ),
    Strategy(
        name="s4-combined",
        display_name="s4-combined",
        filepath=result("s4-combined/java_20_s4_sonnet4.6_cost_3.0_runs_5"),
        model=SONNET_4_6,
    ),
]

In [ ]:
# ── Select model ────────────────────────────────────────────────────────────
# Change this to strategies_sonnet to switch models; then re-run cells below
strategies = None

if DESIRED_MODEL == GPT_5_4:
    strategies = strategies_gpt
elif DESIRED_MODEL == SONNET_4_6:
    strategies = strategies_sonnet
else:
    raise ValueError(
        f"DESIRED_MODEL must be either '{GPT_5_4}' or '{SONNET_4_6}', got '{DESIRED_MODEL}'")

print(f"Active model : {strategies[0].model}")
for s in strategies:
    print(f"  {s.display_name:<25} {s.filepath}")

In [ ]:
# Load metrics + pooled observations for every strategy
all_metrics = {s.name: load_metrics(s.filepath) for s in strategies}
all_obs     = {s.name: load_instance_observations(s.filepath) for s in strategies}

In [ ]:
# ── Pass rate comparison ─────────────────────────────────────────────────────
plot_pass_rate_nway(strategies, all_metrics)
plt.show()

In [ ]:
# ── Pooled cost, API calls & token usage ────────────────────────────────────
plot_pooled_metrics_nway(strategies, all_obs, strategies[0].model)
plt.show()

In [ ]:
# ── Instance resolution table ───────────────────────────────────────────────
# Rows = instance_ids sorted by s0-original resolved count (desc).
# Cells show which runs resolved each instance (e.g. "R1, R3") or "—".
# Last row = total distinct instances resolved per strategy.
df_resolved = build_resolution_table(strategies)

strategy_cols = [s.display_name for s in strategies]

def _color_resolution(col):
    # Explicit bg + text color on every cell to avoid white-on-white in dark themes.
    styles = []
    for v in col:
        if v != "—" and v != "":
            styles.append("background-color: #D6EAD6; color: #1a1a1a")
        else:
            styles.append("background-color: #FAFAFA; color: #1a1a1a")
    return styles

from IPython.display import display
display(
    df_resolved.style
    .set_caption("Instance resolution by strategy  (sorted by s0-original resolved count)")
    .set_properties(**{"text-align": "center", "font-size": "11px", "color": "#1a1a1a"})
    .set_properties(subset=["instance_id"], **{"text-align": "left"})
    .apply(_color_resolution, subset=strategy_cols)
)

In [ ]:
# ── Statistical significance: s0-original vs each other strategy ────────────
# Wilcoxon rank-sum test (α=0.05). Highlighted cells = significant (p<0.05).
# pass_rate: per-run values (N = n_runs); agent metrics: pooled obs (N = runs × instances).
# A12 > 0.5 → s0-original tends to produce larger values.
s0 = next(s for s in strategies if s.name == "s0-original")
comparisons = [
    (s.display_name, all_metrics[s.name], all_obs[s.name])
    for s in strategies if s.name != "s0-original"
]
labels_sX = [c[0] for c in comparisons]

df_sig = build_compact_stat_sig_df(all_metrics[s0.name], all_obs[s0.name], comparisons)
display_compact_stat_sig(df_sig, s0.display_name, labels_sX)